# 03. 멀티 LLM 오케스트레이션 🏪

## 학습 목표
- 여러 LLM을 함께 사용하는 패턴 이해
- 병렬 호출, Fallback, 결과 합산 전략 습득
- Rate Limiting 및 Cost 최적화 기법
- ai-ipsonum 3-engine 파이프라인 가중 앙상블 시뮬레이션

## ai-ipsonum 연계 🏪
- 현재: ChatGPT, Perplexity, Gemini 병렬 쿼리 → 단순 합산
- 개선안: 엔진별 정확도 기반 가중 앙상블
- 참고: `ai-ipsonum/src/lib/ai-engines/`

---

In [ ]:
import asyncio
import json
import time
import random
from collections import Counter
from typing import Optional

import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'DejaVu Sans'
import numpy as np
import pandas as pd

## 1. 멀티 LLM 패턴: 왜 여러 모델을 함께 쓰는가

### 동기

| 이유 | 설명 |
|------|------|
| **신뢰성** | 한 모델의 hallucination을 다른 모델이 보완 |
| **가용성** | 모델 A 장애 시 모델 B로 자동 전환 |
| **비용** | 단순 작업은 저렴한 모델, 복잡한 작업은 고성능 모델 |
| **정확도** | 여러 모델의 결과를 교차 검증 하여 정확도 향상 |

### ai-ipsonum의 현재 구조

```
사용자 쿼리: "성수동 카페 TOP 5"
       │
       ├──→ ChatGPT  ─→ [블루보틀, 스타벅스, 컨테이너, ...]
       ├──→ Perplexity → [블루보틀, 컨테이너, 솔길체, ...]
       └──→ Gemini    ─→ [블루보틀, 스타벅스, 플릿화이트, ...]
       │
       └──→ 합산 ─→ 최종 결과
```

In [ ]:
# --- Mock LLM 엔진 정의 ---

class MockLLMEngine:
    """LLM 엔진 시뮬레이션"""
    
    def __init__(self, name: str, reliability: float, accuracy: float,
                 cost_per_call: float, latency_range: tuple):
        self.name = name
        self.reliability = reliability  # 성공률 (0~1)
        self.accuracy = accuracy        # 정확도 (0~1)
        self.cost_per_call = cost_per_call  # USD
        self.latency_range = latency_range  # (최소, 최대) 초
        self.call_count = 0
        self.total_cost = 0.0
    
    def call(self, query: str) -> dict:
        """LLM 호출 시뮬레이션"""
        self.call_count += 1
        self.total_cost += self.cost_per_call
        
        # 랜덤 실패 시뮬레이션
        if random.random() > self.reliability:
            return {"success": False, "error": f"{self.name} timeout", "latency": self.latency_range[1]}
        
        # 레이턴시 시뮬레이션
        latency = random.uniform(*self.latency_range)
        
        # 결과 생성 (정확도에 따라 다른 결과)
        all_stores = [
            {"name": "블루보틀 성수점", "category": "카페", "is_real": True},
            {"name": "스타벅스 종로점", "category": "카페", "is_real": True},
            {"name": "컨테이너 성수동", "category": "카페", "is_real": True},
            {"name": "솔길체", "category": "카페", "is_real": True},
            {"name": "플릿화이트", "category": "카페", "is_real": True},
            {"name": "매머드 카페", "category": "카페", "is_real": False},  # hallucination
            {"name": "리보 커피", "category": "카페", "is_real": False},      # hallucination
        ]
        
        # 정확도에 따라 실제 매장 선택 비율 조정
        n_results = random.randint(3, 5)
        real_count = int(n_results * self.accuracy)
        fake_count = n_results - real_count
        
        real_stores = [s for s in all_stores if s["is_real"]]
        fake_stores = [s for s in all_stores if not s["is_real"]]
        
        selected = random.sample(real_stores, min(real_count, len(real_stores)))
        if fake_count > 0:
            selected += random.sample(fake_stores, min(fake_count, len(fake_stores)))
        
        random.shuffle(selected)
        
        return {
            "success": True,
            "stores": [{"name": s["name"], "rank": i+1, "category": s["category"]} 
                       for i, s in enumerate(selected)],
            "latency": latency
        }


# 3개 엔진 정의 (ai-ipsonum 구조 반영)
engines = {
    "chatgpt": MockLLMEngine("ChatGPT", reliability=0.95, accuracy=0.85, 
                              cost_per_call=0.03, latency_range=(1.0, 3.0)),
    "perplexity": MockLLMEngine("Perplexity", reliability=0.90, accuracy=0.90,
                                 cost_per_call=0.02, latency_range=(0.5, 2.0)),
    "gemini": MockLLMEngine("Gemini", reliability=0.92, accuracy=0.80,
                             cost_per_call=0.01, latency_range=(0.8, 2.5)),
}

print("=== 엔진 스펙 ===")
for name, engine in engines.items():
    print(f"  {engine.name}: reliability={engine.reliability:.0%}, "
          f"accuracy={engine.accuracy:.0%}, cost=${engine.cost_per_call}")

---
## 2. 병렬 호출: asyncio로 동시 요청

3개 엔진을 **순차적**으로 호출하면 레이턴시가 합산된다.  
**병렬 호출**을 사용하면 가장 느린 엔진의 레이턴시만끼 든다.

$$T_{\text{sequential}} = \sum_{i=1}^{n} t_i \quad vs \quad T_{\text{parallel}} = \max(t_1, t_2, \ldots, t_n)$$

In [ ]:
# 순차 vs 병렬 호출 비교

random.seed(42)
query = "성수동 카페 TOP 5"

# 순차 호출
sequential_results = {}
sequential_latencies = []

for name, engine in engines.items():
    result = engine.call(query)
    sequential_results[name] = result
    sequential_latencies.append(result["latency"])

total_sequential = sum(sequential_latencies)

print("=== 순차 호출 ===")
for name, result in sequential_results.items():
    status = "\u2713" if result["success"] else "\u2717"
    print(f"  {name}: {status} ({result['latency']:.2f}s)")
print(f"  전체 시간: {total_sequential:.2f}s (\ud569\uc0b0)")

# 병렬 호출 (asyncio 시뮬레이션)
random.seed(42)
parallel_results = {}
parallel_latencies = []

for name, engine in engines.items():
    engine.call_count = 0  # 리셋
    result = engine.call(query)
    parallel_results[name] = result
    parallel_latencies.append(result["latency"])

total_parallel = max(parallel_latencies)

print(f"\n=== 병렬 호출 ===")
for name, result in parallel_results.items():
    status = "\u2713" if result["success"] else "\u2717"
    print(f"  {name}: {status} ({result['latency']:.2f}s)")
print(f"  전체 시간: {total_parallel:.2f}s (max)")

print(f"\n\u2192 속도 향상: {total_sequential/total_parallel:.1f}x 빠름")

In [ ]:
# asyncio 병렬 호출 코드 예시 (Google Colab에서 실행 가능)

async def call_engine_async(engine: MockLLMEngine, query: str) -> dict:
    """비동기 엔진 호출 (실제로는 aiohttp 사용)"""
    # 네트워크 지연 시뮬레이션
    latency = random.uniform(*engine.latency_range)
    await asyncio.sleep(latency * 0.01)  # 실제 대신 매우 짧은 슬립
    result = engine.call(query)
    result["engine"] = engine.name
    return result

async def call_all_engines(engines: dict, query: str) -> list[dict]:
    """\ubaa8\ub4e0 \uc5d4\uc9c4\uc744 \ubcd1\ub82c\ub85c \ud638\ucd9c"""
    tasks = [call_engine_async(engine, query) for engine in engines.values()]
    results = await asyncio.gather(*tasks, return_exceptions=True)
    return results

# Colab/Jupyter에서는 await 직접 사용 가능
random.seed(123)
results = await call_all_engines(engines, "성수동 카페 TOP 5")

print("=== asyncio 병렬 호출 결과 ===")
for r in results:
    if isinstance(r, dict):
        status = "\u2713" if r.get("success") else "\u2717"
        engine_name = r.get("engine", "unknown")
        store_count = len(r.get("stores", []))
        print(f"  {engine_name}: {status} - {store_count}개 매장")
    else:
        print(f"  Error: {r}")

---
## 3. Fallback 전략: 모델 A 실패 시 B로

### Fallback 패턴

```
1차: 기본 모델 (\uace0성\ub2a5, \ube44\uc2dc)
   │ 실패 (\ud0c0\uc784\uc544\uc6c3/\uc5d0\ub7ec)
   ▼
2차: 백업 모델 (\uc911간 성능, 매우 매\uc800\ub834\ud55c)
   │ 실패
   ▼
3차: 최종 백업 (저성능이라도 응답 보장)
```

In [ ]:
# Fallback 로직 구현

def call_with_fallback(engines: list[MockLLMEngine], query: str, 
                       timeout: float = 3.0) -> dict:
    """
    Fallback 전략으로 LLM 호출.
    첫 번째 엔진이 실패하면 다음 엔진으로.
    """
    attempts = []
    
    for engine in engines:
        result = engine.call(query)
        
        attempt = {
            "engine": engine.name,
            "success": result["success"],
            "latency": result["latency"]
        }
        
        # 타임아웃 체크
        if result["latency"] > timeout:
            attempt["success"] = False
            attempt["error"] = f"Timeout ({result['latency']:.1f}s > {timeout}s)"
        
        attempts.append(attempt)
        
        if attempt["success"]:
            return {
                "result": result,
                "attempts": attempts,
                "final_engine": engine.name
            }
    
    return {
        "result": None,
        "attempts": attempts,
        "final_engine": None
    }


# Fallback 시뮬레이션
random.seed(7)  # 실패 시나리오를 위한 seed
fallback_order = [
    MockLLMEngine("GPT-4o", reliability=0.3, accuracy=0.95, cost_per_call=0.05, latency_range=(2.0, 5.0)),
    MockLLMEngine("GPT-3.5", reliability=0.95, accuracy=0.75, cost_per_call=0.002, latency_range=(0.5, 1.5)),
    MockLLMEngine("Local-LLM", reliability=0.99, accuracy=0.60, cost_per_call=0.0, latency_range=(0.1, 0.5)),
]

result = call_with_fallback(fallback_order, "성수동 카페", timeout=3.0)

print("=== Fallback 결과 ===")
for attempt in result["attempts"]:
    status = "\u2713" if attempt["success"] else "\u2717"
    error = f" ({attempt.get('error', 'failed')})" if not attempt["success"] else ""
    print(f"  {attempt['engine']}: {status}{error} - {attempt['latency']:.2f}s")
print(f"  최종 엔진: {result['final_engine']}")

---
## 4. 결과 합산 전략: 단순 합산, 투표, 가중 앙상블

3개 엔진의 결과를 하나로 합치는 전략.

| 전략 | 설명 | 수식 |
|------|------|------|
| **단순 합산** | 모든 매장을 합쳐서 중복 제거 | $\text{Union}(R_1, R_2, \ldots)$ |
| **투표 (Voting)** | 여러 엔진이 언급한 매장만 선택 | $\text{count}(s) \geq k$ |
| **가중 앙상블** | 엔진 신뢰도로 가중치 | $\text{score}(s) = \sum w_i \cdot \mathbb{1}[s \in R_i]$ |

In [ ]:
# --- 3개 엔진 결과 샘플 데이터 ---

engine_results = {
    "ChatGPT": [
        {"name": "블루보틀 성수점", "rank": 1, "category": "카페"},
        {"name": "스타벅스 종로점", "rank": 2, "category": "카페"},
        {"name": "컨테이너 성수동", "rank": 3, "category": "카페"},
        {"name": "매머드 카페", "rank": 4, "category": "카페"},  # hallucination
    ],
    "Perplexity": [
        {"name": "블루보틀 성수점", "rank": 1, "category": "카페"},
        {"name": "컨테이너 성수동", "rank": 2, "category": "카페"},
        {"name": "솔길체", "rank": 3, "category": "카페"},
        {"name": "플릿화이트", "rank": 4, "category": "카페"},
    ],
    "Gemini": [
        {"name": "블루보틀 성수점", "rank": 1, "category": "카페"},
        {"name": "스타벅스 종로점", "rank": 2, "category": "카페"},
        {"name": "플릿화이트", "rank": 3, "category": "카페"},
        {"name": "리보 커피", "rank": 4, "category": "카페"},  # hallucination
    ]
}

print("=== 엔진별 결과 ===")
for engine, stores in engine_results.items():
    names = [s["name"] for s in stores]
    print(f"  {engine}: {names}")

In [ ]:
# --- 전략 1: 단순 합산 (Union) ---

def simple_union(engine_results: dict) -> list[str]:
    """모든 엔진 결과를 합쳐서 중복 제거"""
    all_names = set()
    for stores in engine_results.values():
        for store in stores:
            all_names.add(store["name"])
    return sorted(all_names)

# --- 전략 2: 투표 (Voting) ---

def majority_voting(engine_results: dict, min_votes: int = 2) -> list[tuple]:
    """최소 min_votes개 엔진이 언급한 매장만 선택"""
    vote_counter = Counter()
    for stores in engine_results.values():
        for store in stores:
            vote_counter[store["name"]] += 1
    
    return [(name, votes) for name, votes in vote_counter.most_common() 
            if votes >= min_votes]

# --- 전략 3: 가중 쑙상블 ---

def weighted_ensemble(engine_results: dict, weights: dict) -> list[tuple]:
    """엔진별 가중치를 적용하여 점수 계산"""
    scores = {}
    for engine, stores in engine_results.items():
        w = weights.get(engine, 1.0)
        for store in stores:
            name = store["name"]
            # 순위 반영: 높은 순위일수록 높은 점수
            rank_score = 1.0 / store["rank"]
            if name not in scores:
                scores[name] = 0
            scores[name] += w * rank_score
    
    return sorted(scores.items(), key=lambda x: x[1], reverse=True)


# 결과 비교
print("=== 전\ub7b5 1: \ub2e8\uc21c \ud569\uc0b0 ===")
union_result = simple_union(engine_results)
print(f"  {union_result}")
print(f"  \u2192 hallucination \ud3ec\ud568 \uac00\ub2a5\uc131 \ub192\uc74c ({len(union_result)}\uac1c)")

print("\n=== \uc804\ub7b5 2: \ud22c\ud45c (2\ud45c \uc774\uc0c1) ===")
vote_result = majority_voting(engine_results, min_votes=2)
for name, votes in vote_result:
    print(f"  {name}: {votes}\ud45c")
print(f"  \u2192 hallucination \ud544\ud130\ub9c1 \ud6a8\uacfc ({len(vote_result)}\uac1c)")

print("\n=== \uc804\ub7b5 3: \uac00\uc911 \uc559\uc0c1\ube14 ===")
weights = {"ChatGPT": 0.85, "Perplexity": 0.90, "Gemini": 0.80}
ensemble_result = weighted_ensemble(engine_results, weights)
for name, score in ensemble_result:
    print(f"  {name}: {score:.3f}")

In [ ]:
# 전략별 결과 시각화

ground_truth = {"블루보틀 성수점", "스타벅스 종로점", "컨테이너 성수동", "솔길체", "플릿화이트"}

strategies = {
    "Union": set(union_result),
    "Voting (2+)": {name for name, _ in vote_result},
    "Ensemble (top5)": {name for name, _ in ensemble_result[:5]},
}

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, (strategy, predicted) in zip(axes, strategies.items()):
    tp = len(predicted & ground_truth)
    fp = len(predicted - ground_truth)
    fn = len(ground_truth - predicted)
    
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    
    metrics = [precision, recall, f1]
    labels = ["Precision", "Recall", "F1"]
    colors = ["#4ecdc4", "#45b7d1", "#f7dc6f"]
    
    bars = ax.bar(labels, metrics, color=colors, edgecolor='black', linewidth=0.5)
    ax.set_title(f"{strategy}\n(TP={tp}, FP={fp}, FN={fn})", fontsize=11)
    ax.set_ylim(0, 1.2)
    ax.grid(axis='y', alpha=0.3)
    
    for bar, val in zip(bars, metrics):
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.02,
                f'{val:.2f}', ha='center', va='bottom', fontweight='bold')

plt.suptitle("Aggregation Strategy Comparison", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 5. Rate Limiting & Cost 최적화

여러 엔진을 사용하면 **비용이 빠르게 증가**한다.  
토큰 사용량을 추적하고, 비용 예산을 관리하는 것이 중요.

### 비용 최적화 전략

1. **캐싱**: 동일 쿼리는 캐시된 결과 사용
2. **티어 라우팅**: 단순 쿼리는 저렴한 모델, 복잡한 쿼리만 고성능 모델
3. **선택적 호출**: 항상 3개 모두 호출하지 않고, 필요시만

In [ ]:
# Rate Limiter 구현

class RateLimiter:
    """API Rate Limiting 관리"""
    
    def __init__(self, max_calls_per_minute: int, max_tokens_per_minute: int):
        self.max_calls = max_calls_per_minute
        self.max_tokens = max_tokens_per_minute
        self.call_timestamps = []
        self.token_usage = []
    
    def can_call(self, estimated_tokens: int = 100) -> bool:
        """\ud638\ucd9c \uac00\ub2a5 \uc5ec\ubd80 \ud655\uc778"""
        now = time.time()
        # 1\ubd84 \uc774\ub0b4 \ud638\ucd9c \uc218 \ud655\uc778
        recent_calls = [t for t in self.call_timestamps if now - t < 60]
        recent_tokens = sum(t for ts, t in self.token_usage if now - ts < 60)
        
        return len(recent_calls) < self.max_calls and \
               recent_tokens + estimated_tokens < self.max_tokens
    
    def record_call(self, tokens_used: int):
        now = time.time()
        self.call_timestamps.append(now)
        self.token_usage.append((now, tokens_used))


# Cost Tracker
class CostTracker:
    """API 비용 추적"""
    
    def __init__(self, daily_budget: float):
        self.daily_budget = daily_budget
        self.costs = []  # (timestamp, engine, cost)
    
    def record(self, engine: str, cost: float):
        self.costs.append((time.time(), engine, cost))
    
    def total_cost(self) -> float:
        return sum(c for _, _, c in self.costs)
    
    def remaining_budget(self) -> float:
        return self.daily_budget - self.total_cost()
    
    def summary(self) -> dict:
        by_engine = {}
        for _, engine, cost in self.costs:
            by_engine[engine] = by_engine.get(engine, 0) + cost
        return by_engine


# 시뮬레이션: 100건 쿼리 처리
tracker = CostTracker(daily_budget=5.0)  # $5/일

np.random.seed(42)
for i in range(100):
    for name, engine in engines.items():
        tracker.record(engine.name, engine.cost_per_call)

print("=== 100건 쿼리 비용 요약 ===")
print(f"  총 비용: ${tracker.total_cost():.2f}")
print(f"  일\uc77c \uc608\uc0b0: ${tracker.daily_budget:.2f}")
print(f"  \ub0a8\uc740 \uc608\uc0b0: ${tracker.remaining_budget():.2f}")
print(f"\n  \uc5d4\uc9c4\ubcc4 \ube44\uc6a9:")
for engine, cost in tracker.summary().items():
    print(f"    {engine}: ${cost:.2f}")

---
## 6. 🏪 3-engine 파이프라인 개선: 가중 앙상블

### 현재 ai-ipsonum 방식
- 3개 엔진 단순 합산 (Union) → 중복 제거
- 문제: hallucination이 그대로 포함됨

### 개선안
- 엔진별 **정확도 기반 가중치** 적용
- 순위 반영한 점수 계산
- 임계값 이상만 최종 결과에 포함

In [ ]:
# --- 대규모 시뮬레이션: 단순 합산 vs 가중 쑙상블 ---

def simulate_queries(n_queries: int = 200, seed: int = 42) -> pd.DataFrame:
    """여러 쿼리에 대해 두 전략 비교 시뮬레이션"""
    np.random.seed(seed)
    random.seed(seed)
    
    # 실제 매장 데이터베이스
    real_stores_db = [
        "블루보틀 성수점", "스타벅스 종로점", "컨테이너 성수동",
        "솔길체", "플릿화이트", "이디야 커피", "노마드",
        "매베릭스", "라이트프렌즈", "몰티드 시청점"
    ]
    fake_stores = ["매머드 카페", "리보 커피", "드림 카페", "실버 빈"]
    
    # 엔진 정확도 설정
    engine_configs = {
        "ChatGPT": {"accuracy": 0.85, "weight": 0.85},
        "Perplexity": {"accuracy": 0.90, "weight": 0.90},
        "Gemini": {"accuracy": 0.80, "weight": 0.80},
    }
    
    results = []
    
    for q in range(n_queries):
        # 각 쿼리마다 정답 매장 5개 랜덤 선택
        gt = set(random.sample(real_stores_db, 5))
        
        engine_outputs = {}
        for eng_name, config in engine_configs.items():
            n_results = random.randint(3, 5)
            real_count = int(n_results * config["accuracy"])
            fake_count = n_results - real_count
            
            selected_real = random.sample(list(gt), min(real_count, len(gt)))
            selected_fake = random.sample(fake_stores, min(fake_count, len(fake_stores)))
            
            output = []
            for i, name in enumerate(selected_real + selected_fake, 1):
                output.append({"name": name, "rank": i})
            
            engine_outputs[eng_name] = output
        
        # 전략 1: 단순 합산
        union_names = set()
        for stores in engine_outputs.values():
            for s in stores:
                union_names.add(s["name"])
        
        union_tp = len(union_names & gt)
        union_fp = len(union_names - gt)
        union_precision = union_tp / len(union_names) if union_names else 0
        union_recall = union_tp / len(gt) if gt else 0
        
        # 전략 2: 가중 쑙상블 (top 5)
        scores = {}
        for eng_name, stores in engine_outputs.items():
            w = engine_configs[eng_name]["weight"]
            for s in stores:
                name = s["name"]
                rank_score = 1.0 / s["rank"]
                scores[name] = scores.get(name, 0) + w * rank_score
        
        top_ensemble = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:5]
        ensemble_names = {name for name, _ in top_ensemble}
        
        ensemble_tp = len(ensemble_names & gt)
        ensemble_fp = len(ensemble_names - gt)
        ensemble_precision = ensemble_tp / len(ensemble_names) if ensemble_names else 0
        ensemble_recall = ensemble_tp / len(gt) if gt else 0
        
        results.append({
            "query": q,
            "union_precision": union_precision,
            "union_recall": union_recall,
            "union_f1": 2 * union_precision * union_recall / (union_precision + union_recall) if (union_precision + union_recall) > 0 else 0,
            "ensemble_precision": ensemble_precision,
            "ensemble_recall": ensemble_recall,
            "ensemble_f1": 2 * ensemble_precision * ensemble_recall / (ensemble_precision + ensemble_recall) if (ensemble_precision + ensemble_recall) > 0 else 0,
        })
    
    return pd.DataFrame(results)


df = simulate_queries(200)

print("=== 200개 쿼리 시뮬레이션 결과 ===")
print(f"\n{'Metric':<25} {'Union':>10} {'Ensemble':>10} {'Delta':>10}")
print("-" * 55)
for metric in ["precision", "recall", "f1"]:
    u = df[f"union_{metric}"].mean()
    e = df[f"ensemble_{metric}"].mean()
    delta = e - u
    direction = "+" if delta > 0 else ""
    print(f"Avg {metric:<20} {u:>10.4f} {e:>10.4f} {direction}{delta:>9.4f}")

In [ ]:
# 결과 시각화

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, metric in zip(axes, ["precision", "recall", "f1"]):
    union_vals = df[f"union_{metric}"]
    ensemble_vals = df[f"ensemble_{metric}"]
    
    ax.hist(union_vals, bins=20, alpha=0.6, label="Union (current)", color="#ff6b6b")
    ax.hist(ensemble_vals, bins=20, alpha=0.6, label="Ensemble (improved)", color="#4ecdc4")
    
    ax.axvline(union_vals.mean(), color="red", linestyle="--", linewidth=2,
               label=f"Union avg: {union_vals.mean():.3f}")
    ax.axvline(ensemble_vals.mean(), color="teal", linestyle="--", linewidth=2,
               label=f"Ensemble avg: {ensemble_vals.mean():.3f}")
    
    ax.set_title(f"{metric.capitalize()}", fontsize=12, fontweight='bold')
    ax.set_xlabel(metric.capitalize())
    ax.set_ylabel("Count")
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

plt.suptitle("Union vs Weighted Ensemble (200 queries)", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 연습 문제

아래 문제를 직접 풀어보세요.

### 연습 1: 엔진별 가중치 학습

엔진별 가중치를 **지난 쿼리 결과를 바탕으로 자동으로 학습**하는 함수를 구현하세요.

- 입력: 각 엔진의 예측 결과 + 정답 (ground truth)
- 출력: 엔진별 업데이트된 가중치
- 방법: 각 엔진의 F1 점수를 가중치로 사용

In [ ]:
# TODO: 엔진별 가중치 학습 함수 구현
# 요구사항:
# 1. 각 엔진의 예측 결과와 ground truth를 비교
# 2. 엔진별 F1 점수 계산
# 3. F1 점수를 정규화하여 가중치로 사용 (\ud569\uc774 1\uc774 \ub418\ub3c4\ub85d)

# def learn_weights(engine_predictions: dict, ground_truth: set) -> dict:
#     TODO
#     return weights

# \ud14c\uc2a4\ud2b8 \ub370\uc774\ud130
# test_predictions = {
#     "ChatGPT": ["\ube14\ub8e8\ubcf4\ud2c0", "\uc2a4\ud0c0\ubc85\uc2a4", "\ub9e4\uba38\ub4dc \uce74\ud398"],  # 2/3 \uc815\ud655
#     "Perplexity": ["\ube14\ub8e8\ubcf4\ud2c0", "\ucee8\ud14c\uc774\ub108", "\uc194\uae38\uccb4"],     # 3/3 \uc815\ud655
#     "Gemini": ["\ube14\ub8e8\ubcf4\ud2c0", "\ub9ac\ubcf4 \ucee4\ud53c", "\ub4dc\ub9bc \uce74\ud398"],    # 1/3 \uc815\ud655
# }
# test_gt = {"\ube14\ub8e8\ubcf4\ud2c0", "\uc2a4\ud0c0\ubc85\uc2a4", "\ucee8\ud14c\uc774\ub108", "\uc194\uae38\uccb4", "\ud50c\ub9bf\ud654\uc774\ud2b8"}

# weights = learn_weights(test_predictions, test_gt)
# print(weights)

---
## 핵심 정리

| 개념 | 설명 | ai-ipsonum 적용 |
|------|------|-------------|
| 병렬 호출 | asyncio로 동시 요청 | 3개 엔진 동시 호출 |
| Fallback | 실패 시 다음 엔진 | 타임아웃/장애 대응 |
| 단순 합산 | Union + 중복 제거 | 현재 ai-ipsonum 방식 |
| 투표 (Voting) | 다수결로 hallucination 필터 | 간단한 개선 |
| 가중 쑙상블 | 엔진 신뢰도 + 순위 반영 | 최적 개선안 |
| Cost 최적화 | 캐싱, 티어 라우팅 | 예산 관리 |

**다음 노트북**: [04-rag-basics.ipynb](04-rag-basics.ipynb) - RAG 기초